In [1]:
import sys
sys.path.append('../')
import transport_frames.src.graph_builder.criteria as criteria
import transport_frames.src.metrics.indicators as indicators
import transport_frames.src.metrics.grade_territory as grade_territory
import transport_frames.src.graph_builder.graphbuilder as graphbuilder
import momepy
import osmnx as ox
import geopandas as gpd
import shapely
from shapely.geometry import Point
import shapely
from shapely import wkt
import pandas as pd
import networkx as nx
import numpy as np
import json
from dongraphio import DonGraphio, GraphType
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [273]:
def read_json_file(filename):
    with open(filename, 'r') as f:
        json_object = json.load(f)

    result_dict = {}

    for key in json_object.keys():
        if isinstance(json_object[key], str):  # check if the value is a string
            try:
                gdf = gpd.read_file(json_object[key])
                result_dict[key] = gdf
            except:
                try:
                    result_dict[key] = json.loads(json_object[key])
                except json.JSONDecodeError:
                    result_dict[key] = json_object[key]
        else:
            result_dict[key] = json_object[key]

    return result_dict

a = read_json_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/indicators/Тюменская_область/Тюменская_область_indicators_gpsp.json')

print(a.keys()) # check all possible keys


dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'train_stops_availability', 'international_aero_availability', 'local_aero_availability', 'port_availability', 'number_of_bus_routes', 'number_of_bus_stops', 'number_of_fuel_stations', 'number_of_train_stops', 'number_of_international_aero', 'number_of_local_aero', 'number_of_ports', 'train_paths_length'])


In [278]:
a['to_fed_roads']

,id,index,name,layer,to_service,geometry
0,0,0,Ощепковское,Абатский район,33.738315,"MULTIPOLYGON (((70.35363 56.53420, 70.41604 56..."
1,1,1,Назаровское,Абатский район,47.061423,"MULTIPOLYGON (((70.72792 56.52697, 70.72904 56..."
2,2,2,Шевыринское,Абатский район,12.737705,"MULTIPOLYGON (((70.32875 56.18267, 70.32965 56..."
3,3,3,Тушнолобовское,Абатский район,5.909034,"MULTIPOLYGON (((69.89056 56.18638, 69.89624 56..."
4,4,4,Партизанское,Абатский район,21.435673,"MULTIPOLYGON (((70.67055 56.08794, 70.67063 56..."
...,...,...,...,...,...,...
274,274,274,Ишим,Городской округ Ишим,5.066324,"MULTIPOLYGON (((69.37101 56.09274, 69.37101 56..."
275,275,275,Тобольск,Городской округ Тобольск,5.278014,"MULTIPOLYGON (((68.14574 58.18468, 68.14770 58..."
276,276,276,Тюмень,Городской округ Тюмень,4.922663,"MULTIPOLYGON (((65.26049 57.28383, 65.26101 57..."
277,277,277,Ялуторовск,городской округ Ялуторовск,4.928637,"MULTIPOLYGON (((66.21807 56.64639, 66.22329 56..."


In [275]:
aggregated_density = a['aggregated_density'].rename(columns={'density': 'road_density'}) # density
road_length_gdf = a['road_length_gdf'] # reg1_length reg2_length reg3_length	
to_fed_roads = a['to_fed_roads'].rename(columns={'density': 'road_density'}) # to_service
to_region_admin_center = a['to_region_admin_center'].rename(columns={'to_service': 'to_region_admin_center'}) # to_service
connectivity = a['connectivity'].rename(columns={'to_service': 'connectivity'})
# connectivity_public_transport = a['connectivity_public_transport'].rename(columns={'to_service': 'connectivity_public_transport'})
azs_availability = a['azs_availability'].rename(columns={'to_service': 'azs_availability'}) # to_service
number_of_fuel_stations = a['number_of_fuel_stations'].rename(columns={'service_count': 'number_of_fuel_stations'}) # service_count
train_stops_availability = a['train_stops_availability'].rename(columns={'to_service': 'train_stops_availability'}) # to_service
number_of_train_stops = a['number_of_train_stops'].rename(columns={'service_count': 'number_of_train_stops'}) # service_count	
international_aero_availability = a['international_aero_availability'].rename(columns={'to_service': 'international_aero_availability'}) # to_service
number_of_international_aero = a['number_of_international_aero'].rename(columns={'service_count': 'number_of_international_aero'}) # service_count
local_aero_availability = a['local_aero_availability'].rename(columns={'to_service': 'local_aero_availability'}) # to_service
number_of_local_aero = a['number_of_local_aero'].rename(columns={'service_count': 'number_of_local_aero'}) # service_count
number_of_bus_stops = a['number_of_bus_stops'].rename(columns={'service_count': 'number_of_bus_stops'}) # service_count
train_paths_length = a['train_paths_length'].rename(columns={'total_length_km': 'train_paths_length'}) # total_length_km
number_of_bus_routes = a['number_of_bus_routes'].rename(columns={'number_of_routes': 'number_of_bus_routes'}) # number_of_routes
number_of_ports = a['number_of_ports'].rename(columns={'service_count': 'number_of_ports'}) 

In [229]:
train_paths_length

,id,index,name,train_paths_length,geometry
0,0,0,Тульская_область,2371.024053,"POLYGON ((296002.715 5971313.180, 296090.881 5..."


In [30]:
gdfs = [
    aggregated_density, road_length_gdf, to_fed_roads, to_region_admin_center,  azs_availability,
    number_of_fuel_stations, train_stops_availability, number_of_train_stops,
    international_aero_availability, number_of_international_aero, local_aero_availability,
    number_of_local_aero, number_of_bus_stops, train_paths_length, number_of_bus_routes, number_of_ports
]

target_crs = 'EPSG:4326'
for i, gdf in enumerate(gdfs):
    gdfs[i] = gdf.to_crs(target_crs)
    print(f"CRS for gdf {i}: {gdfs[i].crs}")

# Проверка всех преобразованных данных
for gdf in gdfs:
    print(gdf.head())

CRS for gdf 0: EPSG:4326
CRS for gdf 1: EPSG:4326
CRS for gdf 2: EPSG:4326
CRS for gdf 3: EPSG:4326
CRS for gdf 4: EPSG:4326
CRS for gdf 5: EPSG:4326
CRS for gdf 6: EPSG:4326
CRS for gdf 7: EPSG:4326
CRS for gdf 8: EPSG:4326
CRS for gdf 9: EPSG:4326
CRS for gdf 10: EPSG:4326
CRS for gdf 11: EPSG:4326
CRS for gdf 12: EPSG:4326
CRS for gdf 13: EPSG:4326
CRS for gdf 14: EPSG:4326
CRS for gdf 15: EPSG:4326
  id                name  road_density  \
0  0  Краснодарский_край      1018.414   

                                            geometry  
0  POLYGON ((36.53049 45.19920, 36.53270 45.18379...  
  id  index                name  reg1_length   reg2_length   reg3_length  \
0  0      0  Краснодарский_край  3485.151304  19206.410862  69651.115089   

                                            geometry  
0  POLYGON ((36.53049 45.19920, 36.53270 45.18379...  
  id  index                name  to_service  \
0  0      0  Краснодарский_край   19.848271   

                                         

In [276]:
gdfs = [
    aggregated_density, road_length_gdf, to_fed_roads, to_region_admin_center, connectivity,  azs_availability,
    number_of_fuel_stations, train_stops_availability, number_of_train_stops, international_aero_availability, number_of_international_aero, 
    local_aero_availability, number_of_local_aero, number_of_bus_stops, train_paths_length, number_of_bus_routes, number_of_ports
]

# Приведение всех геодатафреймов к одной CRS
target_crs = 'EPSG:4326'
for i, gdf in enumerate(gdfs):
    gdfs[i] = gdf.to_crs(target_crs)

# Создаем уникальный идентификатор для каждой строки в каждом GeoDataFrame
for i, gdf in enumerate(gdfs):
    gdf['unique_id'] = gdf.index

merged_gdf = gdfs[0]

# Объединяем все GeoDataFrame по колонке 'unique_id'
for gdf in gdfs[1:]:
    merged_gdf = merged_gdf.merge(gdf, on='unique_id', how='left', suffixes=('', f'_{gdf.name}'))

# Оставляем только нужные колонки, включая 'unique_id'
columns_to_keep = [
    'unique_id', 'geometry', 'name', 'road_density', 'reg1_length', 'reg2_length', 'reg3_length',
    'to_region_admin_center', 'connectivity', 'azs_availability', 'number_of_fuel_stations',
    'train_stops_availability', 'number_of_train_stops', 'international_aero_availability','number_of_international_aero',
        'local_aero_availability', 'number_of_local_aero',
    'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes', 'number_of_ports'
]

# Фильтруем финальный GeoDataFrame по выбранным колонкам
final_gdf = merged_gdf[columns_to_keep]

# Проверяем результат
final_gdf

,unique_id,geometry,name,road_density,reg1_length,reg2_length,reg3_length,to_region_admin_center,connectivity,azs_availability,...,train_stops_availability,number_of_train_stops,international_aero_availability,number_of_international_aero,local_aero_availability,number_of_local_aero,number_of_bus_stops,train_paths_length,number_of_bus_routes,number_of_ports
0,0,"MULTIPOLYGON (((70.35363 56.53420, 70.41604 56...",Ощепковское,546.034,0.000000,170.755587,91.063841,387.237642,3.745559,29.838300,...,73.612483,0.0,274.810967,0.0,76.295350,0.0,0.0,0.000000,0.0,0.0
1,1,"MULTIPOLYGON (((70.72792 56.52697, 70.72904 56...",Назаровское,193.166,0.000000,46.842710,18.292593,400.560750,3.955885,41.238317,...,86.935600,0.0,288.134083,0.0,89.618467,0.0,0.0,0.000000,0.0,0.0
2,2,"MULTIPOLYGON (((70.32875 56.18267, 70.32965 56...",Шевыринское,613.500,2.428493,101.437129,56.792285,376.589727,3.615178,12.700033,...,47.657733,0.0,264.968867,0.0,66.453250,0.0,0.0,0.000000,0.0,0.0
3,3,"MULTIPOLYGON (((69.89056 56.18638, 69.89624 56...",Тушнолобовское,553.875,42.695035,78.726380,34.897969,333.629402,3.081157,8.706917,...,31.729133,0.0,232.927617,0.0,34.412000,0.0,5.0,0.000000,0.0,0.0
4,4,"MULTIPOLYGON (((70.67055 56.08794, 70.67063 56...",Партизанское,334.199,0.000000,45.795596,34.092604,385.335220,3.760936,21.445517,...,65.729050,0.0,273.714350,0.0,75.198733,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,275,"MULTIPOLYGON (((68.14574 58.18468, 68.14770 58...",Тобольск,4026.144,32.305684,161.897215,772.077603,237.540921,3.449421,5.616208,...,14.179117,3.0,171.975425,0.0,14.833758,0.0,244.0,274.944423,0.0,5.0
276,276,"MULTIPOLYGON (((65.26049 57.28383, 65.26101 57...",Тюмень,5304.032,72.506249,450.740467,3174.702724,0.000000,2.949625,1.974883,...,1.621150,9.0,10.499117,1.0,6.137933,1.0,882.0,460.180282,16.0,0.0
277,277,"MULTIPOLYGON (((66.21807 56.64639, 66.22329 56...",Ялуторовск,7477.500,2.136434,47.818822,313.339983,80.101799,2.484423,2.884067,...,0.908083,3.0,64.204217,0.0,57.347750,0.0,121.0,29.419990,23.0,0.0
278,278,"MULTIPOLYGON (((66.08974 56.48680, 66.10479 56...",Заводоуковский,576.531,104.838981,789.531937,831.490666,117.439789,2.581777,11.479317,...,14.818900,8.0,92.168633,0.0,77.611750,0.0,175.0,189.627190,10.0,0.0


In [ ]:
final_gdf.explore('local_aero_availability')

In [279]:
final_gdf = final_gdf.drop(columns=['unique_id'])

In [280]:
final_gdf

,geometry,name,road_density,reg1_length,reg2_length,reg3_length,to_region_admin_center,connectivity,azs_availability,number_of_fuel_stations,train_stops_availability,number_of_train_stops,international_aero_availability,number_of_international_aero,local_aero_availability,number_of_local_aero,number_of_bus_stops,train_paths_length,number_of_bus_routes,number_of_ports
0,"MULTIPOLYGON (((70.35363 56.53420, 70.41604 56...",Ощепковское,546.034,0.000000,170.755587,91.063841,387.237642,3.745559,29.838300,0.0,73.612483,0.0,274.810967,0.0,76.295350,0.0,0.0,0.000000,0.0,0.0
1,"MULTIPOLYGON (((70.72792 56.52697, 70.72904 56...",Назаровское,193.166,0.000000,46.842710,18.292593,400.560750,3.955885,41.238317,0.0,86.935600,0.0,288.134083,0.0,89.618467,0.0,0.0,0.000000,0.0,0.0
2,"MULTIPOLYGON (((70.32875 56.18267, 70.32965 56...",Шевыринское,613.500,2.428493,101.437129,56.792285,376.589727,3.615178,12.700033,1.0,47.657733,0.0,264.968867,0.0,66.453250,0.0,0.0,0.000000,0.0,0.0
3,"MULTIPOLYGON (((69.89056 56.18638, 69.89624 56...",Тушнолобовское,553.875,42.695035,78.726380,34.897969,333.629402,3.081157,8.706917,1.0,31.729133,0.0,232.927617,0.0,34.412000,0.0,5.0,0.000000,0.0,0.0
4,"MULTIPOLYGON (((70.67055 56.08794, 70.67063 56...",Партизанское,334.199,0.000000,45.795596,34.092604,385.335220,3.760936,21.445517,0.0,65.729050,0.0,273.714350,0.0,75.198733,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,"MULTIPOLYGON (((68.14574 58.18468, 68.14770 58...",Тобольск,4026.144,32.305684,161.897215,772.077603,237.540921,3.449421,5.616208,18.0,14.179117,3.0,171.975425,0.0,14.833758,0.0,244.0,274.944423,0.0,5.0
276,"MULTIPOLYGON (((65.26049 57.28383, 65.26101 57...",Тюмень,5304.032,72.506249,450.740467,3174.702724,0.000000,2.949625,1.974883,77.0,1.621150,9.0,10.499117,1.0,6.137933,1.0,882.0,460.180282,16.0,0.0
277,"MULTIPOLYGON (((66.21807 56.64639, 66.22329 56...",Ялуторовск,7477.500,2.136434,47.818822,313.339983,80.101799,2.484423,2.884067,5.0,0.908083,3.0,64.204217,0.0,57.347750,0.0,121.0,29.419990,23.0,0.0
278,"MULTIPOLYGON (((66.08974 56.48680, 66.10479 56...",Заводоуковский,576.531,104.838981,789.531937,831.490666,117.439789,2.581777,11.479317,8.0,14.818900,8.0,92.168633,0.0,77.611750,0.0,175.0,189.627190,10.0,0.0


In [261]:
final_gdf['name'] = final_gdf['name'].replace('Tyumen Oblast', 'Тюменская область')

In [281]:
final_gdf.to_parquet('Тюменская_область_settlements.parquet')

In [166]:
def indicator_area(citygraph, area_polygons1, area_polygons2, area_polygons3, points, inter, region_capital,
                   fuel, train_stops, international_aero, aero, ports, train_paths, bus_stops, crs=3857):
    """
    This function calculates the various indicators for a specific place based on its characteristics.

    Parameters:
    citygraph - Network graph representing the city.
    area_polygons1 - GeoDataFrame of polygons representing the first set of areas (regions/districts).
    area_polygons2 - GeoDataFrame of polygons representing the second set of areas (regions/districts).
    area_polygons3 - GeoDataFrame of polygons representing the third set of areas (regions/districts).
    points - GeoDataFrame of points of all settlements.
    inter - Intermodal graph of the territory.
    region_capital - A special gdf representing the region's capital.
    fuel - The gdf representing fuel stations.
    train_stops - The gdf representing train stops.
    international_aero - The gdf representing international airports.
    aero - The gdf representing local airports.
    ports - The gdf representing ports.
    train_paths - The gdf representing train paths.
    bus_stops - The gdf representing bus stops.

    Returns:
    Three dictionaries of calculated indicators.
    """

    # Подготовка графов и ребер
    n, e = momepy.nx_to_gdf(graphbuilder.prepare_graph(citygraph))
    ei = momepy.nx_to_gdf(inter)[1]
    bus_routes = ei[ei['type'] == 'bus']

    # Вычисление показателей
    density = e.copy()
    density['density'] = density.apply(lambda row: density_roads(gpd.GeoDataFrame([row], geometry='geometry', crs=crs), e, crs=crs), axis=1)
    road_length = aggregate_road_lengths(e, area_polygons1, crs, reg=True)  # Здесь использовано area_polygons1 как шаблон
    connectivity = get_connectivity(citygraph, points, area_polygons1)
    connectivity_public_transport = get_connectivity(inter, points, area_polygons1, graph_type='intermodal')
    to_fed_roads = aggregation(citygraph, points, area_polygons1, service=get_reg(citygraph, 1), weight='length_meter')
    to_fed_roads['to_service'] = to_fed_roads['to_service'] / 1000

    if not region_capital.empty:
        to_region_admin_center = aggregation(citygraph, points, area_polygons1, service=region_capital, weight='length_meter')
        to_region_admin_center['to_service'] = to_region_admin_center['to_service'] / 1000

    azs_availability = None
    number_of_fuel_stations = None
    if not fuel.empty:
        azs_availability = aggregation(citygraph, points, area_polygons1, service=fuel, weight='time_min')
        azs_availability['to_service'] = azs_availability['to_service']
        number_of_fuel_stations = aggregate_services_by_polygon(fuel, area_polygons1)

    train_stops_availability = None
    number_of_train_stops = None
    if not train_stops.empty:
        train_stops_availability = aggregation(citygraph, points, area_polygons1, service=train_stops, weight='time_min')
        train_stops_availability['to_service'] = train_stops_availability['to_service']
        number_of_train_stops = aggregate_services_by_polygon(train_stops, area_polygons1)

    international_aero_availability = None
    number_of_international_aero = None
    if not international_aero.empty:
        international_aero_availability = aggregation(citygraph, points, area_polygons1, service=international_aero, weight='time_min')
        international_aero_availability['to_service'] = international_aero_availability['to_service']
        number_of_international_aero = aggregate_services_by_polygon(international_aero, area_polygons1)

    local_aero_availability = None
    number_of_local_aero = None
    if not aero.empty:
        local_aero_availability = aggregation(citygraph, points, area_polygons1, service=aero, weight='time_min')
        local_aero_availability['to_service'] = local_aero_availability['to_service']
        number_of_local_aero = aggregate_services_by_polygon(aero, area_polygons1)

    port_availability = None
    number_of_ports = None
    if not ports.empty:
        port_availability = aggregation(citygraph, points, area_polygons1, service=ports, weight='time_min')
        port_availability['to_service'] = port_availability['to_service']
        number_of_ports = aggregate_services_by_polygon(ports, area_polygons1)

    number_of_bus_stops = None
    if not bus_stops.empty:
        number_of_bus_stops = aggregate_services_by_polygon(bus_stops, area_polygons1)

    train_paths_length = None
    if not train_paths.empty:
        train_paths_length = aggregate_road_lengths(train_paths, area_polygons1, crs)

    number_of_bus_routes = aggregate_routes_by_polygon(bus_routes, area_polygons1)

    # Функция для агрегации показателей
    def aggregate_results(area_polygons):
        results = {}
        area_polygons_copy = area_polygons.copy()
        area_polygons_copy['density'] = density['density']
        results['aggregated_density'] = area_polygons_copy
        results['road_length_gdf'] = aggregate_road_lengths(e, area_polygons, crs, reg=True)
        results['connectivity'] = get_connectivity(citygraph, points, area_polygons)
        results['connectivity_public_transport'] = get_connectivity(inter, points, area_polygons, graph_type='intermodal')
        results['to_fed_roads'] = aggregation(citygraph, points, area_polygons, service=get_reg(citygraph, 1), weight='length_meter')
        results['to_fed_roads']['to_service'] = results['to_fed_roads']['to_service'] / 1000

        if not region_capital.empty:
            results['to_region_admin_center'] = aggregation(citygraph, points, area_polygons, service=region_capital, weight='length_meter')
            results['to_region_admin_center']['to_service'] = results['to_region_admin_center']['to_service'] / 1000

        if azs_availability is not None:
            results['azs_availability'] = aggregation(citygraph, points, area_polygons, service=fuel, weight='time_min')
            results['azs_availability']['to_service'] = results['azs_availability']['to_service']
            results['number_of_fuel_stations'] = aggregate_services_by_polygon(fuel, area_polygons)

        if train_stops_availability is not None:
            results['train_stops_availability'] = aggregation(citygraph, points, area_polygons, service=train_stops, weight='time_min')
            results['train_stops_availability']['to_service'] = results['train_stops_availability']['to_service']
            results['number_of_train_stops'] = aggregate_services_by_polygon(train_stops, area_polygons)

        if international_aero_availability is not None:
            results['international_aero_availability'] = aggregation(citygraph, points, area_polygons, service=international_aero, weight='time_min')
            results['international_aero_availability']['to_service'] = results['international_aero_availability']['to_service']
            results['number_of_international_aero'] = aggregate_services_by_polygon(international_aero, area_polygons)

        if local_aero_availability is not None:
            results['local_aero_availability'] = aggregation(citygraph, points, area_polygons, service=aero, weight='time_min')
            results['local_aero_availability']['to_service'] = results['local_aero_availability']['to_service']
            results['number_of_local_aero'] = aggregate_services_by_polygon(aero, area_polygons)

        if port_availability is not None:
            results['port_availability'] = aggregation(citygraph, points, area_polygons, service=ports, weight='time_min')
            results['port_availability']['to_service'] = results['port_availability']['to_service']
            results['number_of_ports'] = aggregate_services_by_polygon(ports, area_polygons)

        if number_of_bus_stops is not None:
            results['number_of_bus_stops'] = aggregate_services_by_polygon(bus_stops, area_polygons)

        if train_paths_length is not None:
            results['train_paths_length'] = aggregate_road_lengths(train_paths, area_polygons, crs)

        results['number_of_bus_routes'] = aggregate_routes_by_polygon(bus_routes, area_polygons)
        
        return results

    # Агрегация результатов для каждого набора полигонов
    indicators1 = aggregate_results(area_polygons1)
    indicators2 = aggregate_results(area_polygons2)
    indicators3 = aggregate_results(area_polygons3)

    return indicators1, indicators2, indicators3
